# Workflow: a simple linear workflow

This is notebook 1 of 3 in the workflow series.

A **workflow** is a lightweight named container that groups and orders
process step IRIs.  Each step is recorded independently using an
x-process-step schema (tensile test, simulation, sample preparation, etc.)
and referenced here by IRI.  The workflow node adds a human-readable
envelope and provenance context.

```
workflow/PMDCo instance  (obo:OBI_0000272)
  rdfs:label ─────────── workflow name
  bfo:BFO_0000051 ──────► step IRI 1  (recorded separately)
                  ──────► step IRI 2
                  ──────► step IRI 3
```

Steps are **not** embedded in the workflow node.  Each step IRI points to
an instance described by its own x-process-step schema and stored elsewhere
in the knowledge graph.

---

## Environment setup

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
python3 -m venv .venv && source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE     = pathlib.Path().resolve()   # schemas/workflow/PMDCo/docs/
WORKFLOW = HERE.parent                # schemas/workflow/PMDCo/

schema = Schema(WORKFLOW)

print("Schema:", "/".join(WORKFLOW.parts[-3:]))

Schema: schemas/workflow/OBI


## Step 1: Describe the workflow

Provide a name and an ordered list of step IRIs.  The step IRIs are the
identifiers of independently recorded step instances — each one was created
using an x-process-step schema and stored in the knowledge graph.

| Field | Required | Description |
|---|---|---|
| `label` | yes | Human-readable workflow name |
| `steps` | yes | Ordered list of process step IRIs (≥ 1) |
| `description` | no | Free-text summary |
| `responsible` | no | IRI of the person accountable for this workflow |
| `id` | no | IRI slug; auto-derived from label if omitted |

In [3]:
workflow_input = {
    "label": "QA workflow, batch A",
    "description": "Three-step quality assurance workflow for a steel sample batch.",
    "steps": [
        "https://example.org/steps/raw-material-inspection-1",
        "https://example.org/steps/tensile-test-1",
        "https://example.org/steps/model-calibration-1",
    ],
}

oold_doc = schema.transform(workflow_input)
print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/workflow/OBI/#v2.0.0",
  "type": "obo:OBI_0000272",
  "id": "workflow-qa-workflow-batch-a",
  "label": "QA workflow, batch A",
  "steps": [
    "https://example.org/steps/raw-material-inspection-1",
    "https://example.org/steps/tensile-test-1",
    "https://example.org/steps/model-calibration-1"
  ],
  "description": "Three-step quality assurance workflow for a steel sample batch."
}


The transform adds the OBI class (`obo:OBI_0000272`), derives the workflow
IRI slug from the name, and stamps `conforms_to` for provenance.
The `steps` array is preserved as-is — it is already an ordered list of IRIs.

## Step 2: Convert to RDF

The OO-LD document is parsed into an RDF graph using the ontology context
from `specs/schema.oold.yaml`.  The `steps` array becomes an RDF ordered
list under `bfo:BFO_0000051` so that step ordering is preserved in the graph.

In [4]:
flat = schema.to_graph(workflow_input)

print(f"Graph contains {len(flat)} triples.\n")
print(flat.serialize(format="turtle"))

Graph contains 11 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<https://example.org/workflow-qa-workflow-batch-a> a obo:OBI_0000272 ;
    rdfs:label "QA workflow, batch A" ;
    obo:BFO_0000051 ( <https://example.org/steps/raw-material-inspection-1> <https://example.org/steps/tensile-test-1> <https://example.org/steps/model-calibration-1> ) ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/workflow/OBI/#v2.0.0> ;
    rdfs:comment "Three-step quality assurance workflow for a steel sample batch." .




## Step 3: Validate with SHACL

The SHACL shape (`specs/shape.ttl`) checks that:
- The workflow node has exactly one `rdfs:label` and `dcterms:conformsTo`
- At least one step is referenced via `bfo:BFO_0000051`

In [5]:
conforms, violations = schema.validate(flat)

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: True


## Step 4: Query the graph

SPARQL retrieves the step list in order.  Because the steps are stored as
an RDF list (`rdf:List`), the `rdf:rest*/rdf:first` property path is used
to traverse the list.

In [6]:
OBI = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")

wf_iri = next(flat.subjects(rdflib.RDF.type, OBI["0000272"]))
print("Workflow IRI:", wf_iri)
print("Label:       ", flat.value(wf_iri, rdflib.RDFS.label))
print()

SPARQL = """
PREFIX obi: <http://purl.obolibrary.org/obo/OBI_>
PREFIX bfo: <http://purl.obolibrary.org/obo/BFO_>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?step
WHERE {
  ?wf a obi:0000272 ;
      bfo:0000051 ?list .
  ?list rdf:rest*/rdf:first ?step .
}
"""

print("Steps (in order):")
for row in flat.query(SPARQL):
    print(" ", row.step)

Workflow IRI: https://example.org/workflow-qa-workflow-batch-a
Label:        QA workflow, batch A

Steps (in order):


  https://example.org/steps/raw-material-inspection-1
  https://example.org/steps/tensile-test-1
  https://example.org/steps/model-calibration-1


## Summary

| Step | What happened |
|---|---|
| 1 | Defined a workflow with three step IRIs as a plain Python dict |
| 2 | The transform added the OBI class, derived the workflow IRI, and stamped provenance |
| 3 | The OO-LD was parsed into 11 RDF triples (workflow node + ordered step list) |
| 4 | SHACL validation confirmed structural correctness |
| 5 | SPARQL retrieved the ordered step list from the graph |

Continue with [Notebook 2](2_workflow_branching.ipynb) to see how parallel
branches and custom step ordering work at the step level.

---

## Further reading

- [OO-LD primer](../../../docs/2_oold-primer.md)
- [Schema format reference](../../../docs/3_schema-format.md)
- [Schema patterns guide](../../../docs/4_schema-patterns.md)